# Week 5 Day 2

## Data Cleaning

This notebook demonstrates handling missing dates, missing values,
data type validation, and outlier detection.

In [34]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy import stats

In [35]:
reliance = yf.download(
    "RELIANCE.NS",
    start="2020-01-01",
    end="2025-01-01",
    auto_adjust=True
)

reliance.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,,
2020-01-01,672.216187,680.008852,670.390532,675.956671,14004468
2020-01-02,683.660278,686.176208,673.284921,673.284921,17710316
2020-01-03,684.484070,686.487896,678.183128,682.636062,20984698
2020-01-06,668.609375,680.365106,667.050830,676.847322,24519177
2020-01-07,678.895630,683.304036,673.952826,676.401934,16683622


## Initial Data Inspection

Checking structure, data types and missing values.

In [36]:
reliance.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1238 entries, 2020-01-01 to 2024-12-31
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   (Close, RELIANCE.NS)   1238 non-null   float64
 1   (High, RELIANCE.NS)    1238 non-null   float64
 2   (Low, RELIANCE.NS)     1238 non-null   float64
 3   (Open, RELIANCE.NS)    1238 non-null   float64
 4   (Volume, RELIANCE.NS)  1238 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 58.0 KB


In [37]:
reliance.describe()

Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
count,1238.000000,1238.000000,1238.000000,1238.000000,1.238000e+03
mean,1095.098186,1107.006454,1084.237080,1095.962056,1.951391e+07
std,230.477780,230.698653,230.092008,230.365316,1.612248e+07
min,393.662384,423.029534,389.921900,407.978572,1.705656e+06
25%,951.000778,962.121189,942.435874,951.571376,1.009199e+07
50%,1115.586792,1127.810013,1106.010392,1116.995418,1.425293e+07
75%,1219.325256,1232.958643,1207.933291,1219.190611,2.203644e+07
max,1581.824463,1589.630354,1566.607971,1585.332124,1.426834e+08


In [38]:
reliance.isnull().sum()

Price   Ticker     
Close   RELIANCE.NS    0
High    RELIANCE.NS    0
Low     RELIANCE.NS    0
Open    RELIANCE.NS    0
Volume  RELIANCE.NS    0
dtype: int64

### Observation

No missing values were found in the downloaded dataset.

In [39]:
full_dates = pd.date_range(
    start=reliance.index.min(),
    end=reliance.index.max(),
    freq="D"
)

reliance_full = reliance.reindex(full_dates)

In [40]:
print("Missing dates:")
print(reliance_full.isnull().sum())

Missing dates:
Price   Ticker     
Close   RELIANCE.NS    589
High    RELIANCE.NS    589
Low     RELIANCE.NS    589
Open    RELIANCE.NS    589
Volume  RELIANCE.NS    589
dtype: int64


In [41]:
reliance_ffill = reliance_full.ffill()

In [42]:
reliance_ffill.isnull().sum()

Price   Ticker     
Close   RELIANCE.NS    0
High    RELIANCE.NS    0
Low     RELIANCE.NS    0
Open    RELIANCE.NS    0
Volume  RELIANCE.NS    0
dtype: int64

### Cleaning Decision

Missing dates were introduced after creating a continuous daily calendar.
Values were forward-filled because stock prices remain unchanged on
non-trading days until the next market session.

In [43]:
reliance_ffill.dtypes

Price   Ticker     
Close   RELIANCE.NS    float64
High    RELIANCE.NS    float64
Low     RELIANCE.NS    float64
Open    RELIANCE.NS    float64
Volume  RELIANCE.NS    float64
dtype: object

In [44]:
print(reliance_ffill.columns)

MultiIndex([( 'Close', 'RELIANCE.NS'),
            (  'High', 'RELIANCE.NS'),
            (   'Low', 'RELIANCE.NS'),
            (  'Open', 'RELIANCE.NS'),
            ('Volume', 'RELIANCE.NS')],
           names=['Price', 'Ticker'])


IQR = Q3 - Q1

Lower = Q1 - 1.5×IQR

Upper = Q3 + 1.5×IQR

In [45]:
Q1 = reliance_ffill["Close"].quantile(0.25)
Q3 = reliance_ffill["Close"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

In [46]:
iqr_outliers = reliance_ffill[
    (reliance_ffill["Close"] < lower)
    |
    (reliance_ffill["Close"] > upper)
]

iqr_outliers.head()

Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
2020-01-01,NaN,NaN,NaN,NaN,NaN
2020-01-02,NaN,NaN,NaN,NaN,NaN
2020-01-03,NaN,NaN,NaN,NaN,NaN
2020-01-04,NaN,NaN,NaN,NaN,NaN
2020-01-05,NaN,NaN,NaN,NaN,NaN


In [47]:
print("IQR Outliers:", len(iqr_outliers))

IQR Outliers: 1827


In [48]:
z_scores = np.abs(
    stats.zscore(reliance_ffill["Close"])
)

In [49]:
z_outliers = reliance_ffill[z_scores > 3]

z_outliers.head()

Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
2020-03-23,393.662384,423.029534,389.9219,407.978572,40670949.0


In [50]:
print("Z-Score Outliers:", len(z_outliers))

Z-Score Outliers: 1


In [51]:
print("IQR Outliers:", len(iqr_outliers))
print("Z Outliers:", len(z_outliers))

IQR Outliers: 1827
Z Outliers: 1


### Observation

IQR and Z-score methods identified different numbers of outliers.
This is expected because the methods use different statistical assumptions.

In [52]:
reliance.describe()

Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
count,1238.000000,1238.000000,1238.000000,1238.000000,1.238000e+03
mean,1095.098186,1107.006454,1084.237080,1095.962056,1.951391e+07
std,230.477780,230.698653,230.092008,230.365316,1.612248e+07
min,393.662384,423.029534,389.921900,407.978572,1.705656e+06
25%,951.000778,962.121189,942.435874,951.571376,1.009199e+07
50%,1115.586792,1127.810013,1106.010392,1116.995418,1.425293e+07
75%,1219.325256,1232.958643,1207.933291,1219.190611,2.203644e+07
max,1581.824463,1589.630354,1566.607971,1585.332124,1.426834e+08


In [53]:
reliance_ffill.describe()

Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
count,1827.000000,1827.000000,1827.000000,1827.000000,1.827000e+03
mean,1095.289105,1107.077286,1084.242312,1095.730550,1.986048e+07
std,231.900351,232.144327,231.648779,231.833612,1.656828e+07
min,393.662384,423.029534,389.921900,407.978572,1.705656e+06
25%,951.828979,962.833268,942.522219,953.465098,1.020489e+07
50%,1116.419922,1128.559837,1106.953353,1117.552273,1.451835e+07
75%,1219.958557,1233.308616,1210.156437,1219.296219,2.218511e+07
max,1581.824463,1589.630354,1566.607971,1585.332124,1.426834e+08


## Summary

1. Missing dates were identified by creating a complete daily calendar.
2. Missing values were handled using forward-fill.
3. Data types were validated.
4. Outliers were detected using both IQR and Z-score methods.
5. Summary statistics were reviewed before and after cleaning.